In [1]:
import seaborn             as sns
import os

from pymatgen.core.surface   import generate_all_slabs
from pymatgen.core.structure import Structure
from pymatgen.io.ase         import AseAtomsAdaptor
from ase.io.vasp             import write_vasp

from libraries import surfaces, thermal, vasp_files

sns.set_theme()

/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


# Generate slabs

**Input convention for the bulk structure** (`path_to_structure`):

| file name | meaning | what happens |
|---|---|---|
| `POSCAR*`  | raw structure (database, DFT, ...) | relaxed here with the ML-IAP (**ions and cell**), so the bulk energy is the ML-IAP minimum |
| `CONTCAR*` | already relaxed somehow | used as it is |

`relax_bulk=True/False` overrides the convention. Giving a `CONTCAR` is only consistent if it was relaxed at the
same level of theory that will evaluate the slabs: a DFT geometry evaluated with the ML-IAP is *not* at the ML-IAP
minimum and biases every surface energy computed from it. When the ranking is done with the ML-IAP, give a `POSCAR`
even if it comes from DFT, or give a `CONTCAR` produced by the same ML-IAP. When every energy (bulk and slabs) comes
from DFT (`vasprun.xml`), give the DFT-relaxed `CONTCAR`.

In [ ]:
# Define name of folder and path to the bulk structure
general_folder    = '/Users/cibran/Desktop/estruturas/BiSeBr'
path_to_structure = '/Users/cibran/Desktop/estruturas/BiSeBr/POSCAR'   # 'POSCAR' -> relaxed with the ML-IAP here; 'CONTCAR' -> already relaxed, used as is
relax_bulk        = True  # None: infer from the file name; True/False: force

# ML-IAP used to relax the bulk (and later the slabs)
model_load_path = 'mace-mpa-0-medium.model'
device          = 'cpu'    # switch to 'cuda' if a GPU is available

# Defien KPOINTS density
kpoints_density = [40, 40, 40]  # From reference relaxation

# Maximum number for the Miller index in each direction
max_index = 2

# Whether to repair terminations with broken bonds or just omit them
repair = True

# Whether pymatgen should only keep symmetric slabs (equivalent top and bottom terminations).
# Asymmetric slabs are kept by default: their surface energy is the average of the two terminations.
symmetrize = False

# Minimum thicknesses for slab and vacuum
min_slab_size   = 20.0  # Angstroms
min_vacuum_size = 20.0  # Angstroms

# Finite temperature (optional): besides the 0 K slabs, prepare NPT molecular-dynamics folders (md/<T>K) for the bulk and
# for every slab, to rank the surfaces by free energy at a chosen temperature (last section of this notebook).
finite_temperature = True
md_temperatures    = [300]   # K, temperatures of the NPT runs (the ranking can later be evaluated at any temperature)
md_min_length      = 20.0    # Angstroms, minimum length of the lattice vectors of the MD cell (a and b for slabs, a, b and c for the bulk)
md_n_steps         = 20000   # NPT steps
md_timestep        = 1.0     # fs
md_loginterval     = 10      # a frame is stored every md_loginterval steps (frame spacing = md_loginterval * md_timestep fs)
md_ttime           = 50.0    # fs, thermostat time constant
md_ptime           = 500.0   # fs, barostat time constant
md_equil_ps        = 2.0     # ps discarded at the start of every run (thermalisation) when it is analysed; stored in md.json
md_fix_cell        = None    # None: fully flexible cell; 'all' fixes every lattice vector, 'z' or 'xy' one direction (see mlip.cell_mask)

The surface energy of formation ($\gamma$) is built from free (Gibbs) energies: the free energy of the slab ($G_{slab}$) and the
free energy of the bulk per formula unit ($g_{bulk}$) times the number of formula units in the slab ($N$), per unit of surface area ($A$):

\begin{equation}
    \gamma = \frac{G_{slab} - N g_{bulk}}{2 A}
\end{equation}

The expression is the same at every level; what changes is how $G$ is evaluated. At **0 K** (this notebook and the first part of
`iterate-slabs`) $G = E_0$, the total energy of the relaxed structure, with no zero-point energy and no entropy. At finite temperature
(optional, last section of `iterate-slabs`) $G(T) = U_0(T) + F_{vib}(T)$ from NPT runs. $G$ is a sum of
contributions (no $PV$ term so far), and further ones can be added without changing the formula for $\gamma$.

Free energies are in eV, $A$ is in Å², and $\gamma$ is reported in **J/m²** (1 eV/Å² = 16.0218 J/m²).
$N g_{bulk}$ is implemented as the bulk free energy per atom times the number of atoms of the slab, which is the same
for stoichiometric slabs. The factor of 2 assumes equivalent top and bottom terminations; for asymmetric slabs
$\gamma$ is the average of both. Non-stoichiometric slabs would need explicit chemical potentials.
Each slab stores its `is_symmetric`, `is_polar` and `is_stoichiometric` flags so that these cases can be identified.

In [3]:
# The folder is named slab_v{i}
os.makedirs(general_folder, exist_ok=True)

i = 0
while os.path.exists(f'{general_folder}/slab_v{i}'):
    i += 1
slab_folder = f'{general_folder}/slab_v{i}'
os.makedirs(slab_folder)
slab_folder

'/Users/cibran/Desktop/estruturas-10/BiSeBr/slab_v0'

# Bulk structure

In [4]:
path_to_bulk = f'{slab_folder}/bulk'

# POSCAR -> relax with the ML-IAP (ions and cell); CONTCAR -> use as it is. Writes bulk/POSCAR and bulk/relaxation/{POSCAR,CONTCAR}.
print(surfaces.prepare_bulk_structure(path_to_structure, path_to_bulk, relax=relax_bulk,
                                 model_load_path=model_load_path, device=device))

# Slabs are always cut from the final (relaxed) bulk structure
structure = Structure.from_file(f'{path_to_bulk}/relaxation/CONTCAR')

# Save KPOINTS with the desired density
vasp_files.generate_kpoints(poscar_file=f'{path_to_bulk}/relaxation/CONTCAR', kpoints_file=f'{path_to_bulk}/KPOINTS', kpoints_density=kpoints_density)
vasp_files.prepare_relaxation_folder(path_to_bulk)   # copies KPOINTS (and the other inputs) into bulk/relaxation

# Save bulk information
surfaces.save_json(structure, data='bulk', filename=f'{path_to_bulk}/slab_data.json')

Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/Users/cibran/work/UPC/SurfAIcing/.venv/lib/python3.13/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


      Step     Time          Energy          fmax
BFGS:    0 18:53:12      -42.354028        0.612595
BFGS:    1 18:53:13      -42.368602        0.584448
BFGS:    2 18:53:13      -42.485272        0.302663
BFGS:    3 18:53:13      -42.501817        0.325266
BFGS:    4 18:53:13      -42.552957        0.308178
BFGS:    5 18:53:13      -42.589058        0.218821
BFGS:    6 18:53:13      -42.608655        0.214835
BFGS:    7 18:53:13      -42.616164        0.190964
BFGS:    8 18:53:13      -42.626582        0.182817
BFGS:    9 18:53:13      -42.640030        0.187788
BFGS:   10 18:53:13      -42.654007        0.168689
BFGS:   11 18:53:13      -42.662176        0.113774
BFGS:   12 18:53:13      -42.665111        0.080009
BFGS:   13 18:53:13      -42.666909        0.079418
BFGS:   14 18:53:13      -42.670658        0.094976
BFGS:   15 18:53:13      -42.674957        0.085962
BFGS:   16 18:53:14      -42.678535        0.075057
BFGS:   17 18:53:14      -42.680763        0.090855
BFGS:   18 18:

# Slab generation

Every folder written here (the bulk and each slab) has the same layout, which the other notebooks read:

```
<folder>/POSCAR, KPOINTS     inputs (POTCAR, INCAR, run.sh too, if you add them): the structure and what the calculation needs
<folder>/slab_data.json      Miller index, shift, area, flags (only here: it is not copied anywhere else)
<folder>/relaxation/         copies of the inputs; the 0 K relaxation or DFT run goes here (CONTCAR, vasprun.xml, ...)
<folder>/md/<T>K/            NPT runs (optional, see the last section)
```

In [5]:
# Generate all slabs
slabs = generate_all_slabs(structure,
                           max_index=max_index,
                           min_slab_size=min_slab_size,
                           min_vacuum_size=min_vacuum_size,
                           repair=repair,
                           symmetrize=symmetrize)

In [6]:
n_stoichiometric = 0
for n, slab in enumerate(slabs):
    stoichiometric = slab.composition.reduced_formula == structure.composition.reduced_formula
    n_stoichiometric += stoichiometric
    print(n, 'Polar:', slab.is_polar(), 'Symmetric:', slab.is_symmetric(), 'Stoichiometric:', stoichiometric)

if n_stoichiometric < len(slabs):
    print(f'\n[warning] {len(slabs) - n_stoichiometric} of {len(slabs)} slabs are not stoichiometric: their bulk-referenced '
          f'surface energy depends on the chemical potentials and is not directly comparable.')

0 Polar: False Symmetric: True Stoichiometric: True
1 Polar: False Symmetric: True Stoichiometric: True
2 Polar: False Symmetric: False Stoichiometric: True
3 Polar: False Symmetric: False Stoichiometric: True
4 Polar: False Symmetric: True Stoichiometric: True
5 Polar: False Symmetric: True Stoichiometric: True
6 Polar: False Symmetric: False Stoichiometric: True
7 Polar: False Symmetric: False Stoichiometric: True
8 Polar: False Symmetric: True Stoichiometric: True
9 Polar: False Symmetric: True Stoichiometric: True
10 Polar: False Symmetric: False Stoichiometric: True
11 Polar: False Symmetric: False Stoichiometric: True
12 Polar: False Symmetric: True Stoichiometric: True
13 Polar: False Symmetric: False Stoichiometric: True
14 Polar: False Symmetric: False Stoichiometric: True
15 Polar: False Symmetric: True Stoichiometric: True
16 Polar: False Symmetric: True Stoichiometric: True
17 Polar: False Symmetric: False Stoichiometric: True
18 Polar: False Symmetric: False Stoichiometric

In [7]:
# Iterate over slabs
for i, slab in enumerate(slabs):
    print()
    print(f'Slab {i+1}')
    print()
    print(f'Miller index: {slab.miller_index}')
    print(f'Shift: {slab.shift:.3g}')
    print(f'Surface area: {slab.surface_area:.3g}')
    print(f'Number of sites: {len(slab.sites)}')

    # Miller index tuple to string
    miller_index_str = '_'.join(str(element) for element in slab.miller_index)
    miller_index_str = f'{miller_index_str}_i_{i}'

    # Define current folder
    miller_folder = f'{slab_folder}/{miller_index_str}'

    # Generate new folder
    os.makedirs(miller_folder)

    # Save slab structure into miller_folder
    write_vasp(f'{miller_folder}/POSCAR', AseAtomsAdaptor.get_atoms(slab), direct=True, sort=True)

    # Save KPOINTS with the desired density
    vasp_files.generate_kpoints(poscar_file=f'{miller_folder}/POSCAR', kpoints_file=f'{miller_folder}/KPOINTS', kpoints_density=kpoints_density)

    # Save slab information (including the symmetry, polarity and stoichiometry flags)
    surfaces.save_json(slab, filename=f'{miller_folder}/slab_data.json', bulk=structure)

    # Run folder of the 0 K calculation: copies of POSCAR, KPOINTS, ... (slab_data.json stays in the root)
    vasp_files.prepare_relaxation_folder(miller_folder)


Slab 1

Miller index: (1, 0, 0)
Shift: 0.5
Surface area: 96.8
Number of sites: 36

Slab 2

Miller index: (0, 1, 0)
Shift: 1.51e-14
Surface area: 47.7
Number of sites: 24

Slab 3

Miller index: (0, 1, 0)
Shift: 0.0735
Surface area: 47.7
Number of sites: 24

Slab 4

Miller index: (0, 1, 0)
Shift: 0.148
Surface area: 47.7
Number of sites: 24

Slab 5

Miller index: (0, 1, 0)
Shift: 0.25
Surface area: 47.7
Number of sites: 24

Slab 6

Miller index: (0, 0, 1)
Shift: 3.77e-14
Surface area: 35.5
Number of sites: 12

Slab 7

Miller index: (0, 0, 1)
Shift: 0.0822
Surface area: 35.5
Number of sites: 12

Slab 8

Miller index: (0, 0, 1)
Shift: 0.169
Surface area: 35.5
Number of sites: 12

Slab 9

Miller index: (0, 0, 1)
Shift: 0.25
Surface area: 35.5
Number of sites: 12

Slab 10

Miller index: (1, 1, 0)
Shift: 5.55e-14
Surface area: 108
Number of sites: 36

Slab 11

Miller index: (1, 1, 0)
Shift: 0.102
Surface area: 108
Number of sites: 36

Slab 12

Miller index: (1, 1, 0)
Shift: 0.176
Surface are

# Finite-temperature runs (optional)

With `finite_temperature = True` the bulk and every slab get one folder `md/<T>K` per temperature in `md_temperatures`. Each holds

* `POSCAR`: the structure of its folder, **not relaxed** (the NPT run takes care of that), repeated until its lattice vectors
  are at least `md_min_length` long: only $a$ and $b$ for a slab (the thickness and the vacuum stay as generated), all
  three for the bulk, so that the finite-size error of the bulk and of the slabs is comparable;
* `md.json`: the plan of the run, the supercell (from which the per-atom quantities are scaled back to the slab cell) and
  the thermalisation time `md_equil_ps`, the picoseconds discarded at the start of the run when it is analysed. It is an
  analysis parameter: changing it and running this cell again updates the folders that already exist.

**These cells only prepare the folders; no simulation is run here.** The runs are started in `iterate-slabs` (`finite_temperature = True`
and `run_md = True`, on the same `slab_v<i>` folder) or with VASP.

The simulations are NPT with a **fully flexible cell** unless `md_fix_cell` freezes some of it (stored in `md.json`; the hydrogen
adsorption configurations of `H-absorption` inherit it, together with the other settings of these runs, so it is defined only here).
They can be run in `iterate-slabs`
with the ML-IAP, or with VASP (put `vasprun.xml` in the folder; set `ISIF = 3`, an NPT thermostat/barostat and store every step).

In [8]:
if finite_temperature:
    md_settings = dict(min_length=md_min_length, n_steps=md_n_steps, timestep=md_timestep, loginterval=md_loginterval,
                       ttime=md_ttime, ptime=md_ptime, equil_ps=md_equil_ps, fix_cell=md_fix_cell)
    slab_names = sorted(name for name in os.listdir(slab_folder) if name != 'bulk' and os.path.isdir(f'{slab_folder}/{name}'))

    for name, kind in [('bulk', 'bulk')] + [(name, 'slab') for name in slab_names]:
        md_folders = thermal.prepare_md_folders(f'{slab_folder}/{name}', kind, md_temperatures, **md_settings)
        plan = thermal.read_md_json(md_folders[0])
        print(f"{name}: {plan['n_atoms']} atoms in a {plan['supercell']} supercell of the {plan['n_atoms_cell']}-atom cell, "
              f"{len(md_folders)} temperature(s)")
else:
    print('finite_temperature = False: only the 0 K slabs were generated.')

bulk: 72 atoms in a [3, 2, 1] supercell of the 12-atom cell, 1 temperature(s)
0_0_1_i_5: 72 atoms in a [3, 2, 1] supercell of the 12-atom cell, 1 temperature(s)
0_0_1_i_6: 72 atoms in a [3, 2, 1] supercell of the 12-atom cell, 1 temperature(s)
0_0_1_i_7: 72 atoms in a [3, 2, 1] supercell of the 12-atom cell, 1 temperature(s)
0_0_1_i_8: 72 atoms in a [3, 2, 1] supercell of the 12-atom cell, 1 temperature(s)
0_1_0_i_1: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_0_i_2: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_0_i_3: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_0_i_4: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_1_i_17: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_1_i_18: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_1_i_19: 72 atoms in a [3, 1, 1] supercell of the 24-atom cell, 1 temperature(s)
0_1_1_